# BioPhysTCR Data Preparation & Verification

This notebook verifies the data is correctly loaded and creates train/val splits for training.

**Run this notebook first before training!**

## 1. Environment Setup

In [ ]:
!pip install -r requirements.txt

In [ ]:
import sys
import os

# Set project root (adjust for RunPod)
PROJECT_ROOT = '/workspace/BioPhysTCR'  # RunPod path
# PROJECT_ROOT = '..'  # Local path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import pickle
import json
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Check Data Directory Structure

In [ ]:
data_dir = Path('data')
processed_dir = data_dir / 'processed'
splits_dir = data_dir / 'splits'

print("Data directory structure:")
print("=" * 50)

for item in sorted(processed_dir.rglob('*')):
    if item.is_file():
        size = item.stat().st_size / 1e6
        rel_path = item.relative_to(data_dir)
        print(f"  {rel_path}: {size:.1f} MB")

## 3. Load and Inspect Features

In [ ]:
# Load combined features
combined_path = processed_dir / 'combined_features.pkl'

if combined_path.exists():
    with open(combined_path, 'rb') as f:
        combined = pickle.load(f)
    
    print("Combined features loaded!")
    print(f"Top-level keys: {list(combined.keys())}")
else:
    print(f"WARNING: {combined_path} not found!")
    combined = {}

In [ ]:
# Inspect ESM2 embeddings
esm2 = combined.get('esm2', {})

print("ESM2 Embeddings:")
print("-" * 30)

if 'cdr3' in esm2:
    cdr3_emb = esm2['cdr3']
    print(f"  CDR3 sequences: {len(cdr3_emb)}")
    if cdr3_emb:
        sample_key = list(cdr3_emb.keys())[0]
        sample_emb = cdr3_emb[sample_key]
        if isinstance(sample_emb, np.ndarray):
            print(f"  Sample shape: {sample_emb.shape}")
        elif isinstance(sample_emb, torch.Tensor):
            print(f"  Sample shape: {sample_emb.shape}")

if 'epitope' in esm2:
    epitope_emb = esm2['epitope']
    print(f"  Epitope sequences: {len(epitope_emb)}")

In [ ]:
# Inspect SaProt embeddings
saprot = combined.get('saprot', {})

print("SaProt Embeddings:")
print("-" * 30)
print(f"  Structures: {len(saprot)}")

if saprot:
    sample_key = list(saprot.keys())[0]
    sample_data = saprot[sample_key]
    print(f"  Sample key: {sample_key}")
    print(f"  Sample type: {type(sample_data)}")
    
    if isinstance(sample_data, np.ndarray):
        print(f"  Sample shape: {sample_data.shape}")
    elif isinstance(sample_data, dict):
        print(f"  Sample keys: {list(sample_data.keys())[:5]}")

In [ ]:
# Inspect physicochemical features
physchem = combined.get('physicochemical', {})

print("Physicochemical Features:")
print("-" * 30)
print(f"  Keys: {list(physchem.keys())}")

by_structure = physchem.get('by_structure', {})
print(f"  Structures with features: {len(by_structure)}")

if by_structure:
    sample_key = list(by_structure.keys())[0]
    sample_data = by_structure[sample_key]
    print(f"  Sample key: {sample_key}")
    print(f"  Sample type: {type(sample_data)}")
    
    if isinstance(sample_data, np.ndarray):
        print(f"  Sample shape: {sample_data.shape}")

## 4. Check Graph Data

In [ ]:
# Check graph structures
graphs_path = processed_dir / 'complex_features' / 'complex_graphs.json'

if graphs_path.exists():
    with open(graphs_path, 'r') as f:
        graphs = json.load(f)
    
    print(f"Graph structures loaded: {len(graphs)}")
    
    if graphs:
        sample_key = list(graphs.keys())[0]
        sample_graph = graphs[sample_key]
        print(f"Sample graph: {sample_key}")
        print(f"  Keys: {list(sample_graph.keys())}")
        print(f"  Nodes: {len(sample_graph.get('nodes', []))}")
        print(f"  Edges: {len(sample_graph.get('edges', []))}")
else:
    print(f"WARNING: {graphs_path} not found!")
    graphs = {}

## 5. Create Data Splits

In [ ]:
# Load or create pairs data
pairs_path = processed_dir / 'pairs.json'

if pairs_path.exists():
    with open(pairs_path, 'r') as f:
        pairs = json.load(f)
    print(f"Loaded {len(pairs)} pairs from {pairs_path}")
else:
    # Create pairs from available features
    print("Creating pairs from features...")
    
    pairs = []
    
    # Get all CDR3 sequences and epitopes
    cdr3_seqs = list(esm2.get('cdr3', {}).keys())
    epitope_seqs = list(esm2.get('epitope', {}).keys())
    pdb_ids = list(saprot.keys())
    
    print(f"  CDR3 sequences: {len(cdr3_seqs)}")
    print(f"  Epitope sequences: {len(epitope_seqs)}")
    print(f"  PDB structures: {len(pdb_ids)}")
    
    # Create positive pairs (you would need binding data here)
    # For now, create sample data
    for pdb_id in pdb_ids[:100]:  # Limit for demo
        if cdr3_seqs and epitope_seqs:
            pairs.append({
                'pdb_id': pdb_id,
                'cdr3': cdr3_seqs[0] if cdr3_seqs else '',
                'epitope': epitope_seqs[0] if epitope_seqs else '',
                'label': 1  # Positive binding
            })
    
    print(f"Created {len(pairs)} pairs")

In [ ]:
from sklearn.model_selection import train_test_split

# Scenario 1: Random 90/10 split
train_pairs, val_pairs = train_test_split(
    pairs, 
    test_size=0.1, 
    random_state=42,
    stratify=[p.get('label', 1) for p in pairs] if len(pairs) > 1 else None
)

print(f"Scenario 1 (Random Split):")
print(f"  Train: {len(train_pairs)}")
print(f"  Val: {len(val_pairs)}")

# Save splits
splits_dir.mkdir(parents=True, exist_ok=True)

splits_random = {
    'train': train_pairs,
    'val': val_pairs,
    'test': val_pairs
}

with open(splits_dir / 'splits_random.json', 'w') as f:
    json.dump(splits_random, f, indent=2)
    
print(f"Saved to {splits_dir / 'splits_random.json'}")

In [ ]:
# Scenario 2: Unseen epitopes split
epitopes = list(set(p.get('epitope', '') for p in pairs))
print(f"Unique epitopes: {len(epitopes)}")

if len(epitopes) > 1:
    train_epitopes, val_epitopes = train_test_split(
        epitopes,
        test_size=0.2,
        random_state=42
    )
    
    train_pairs_epi = [p for p in pairs if p.get('epitope', '') in train_epitopes]
    val_pairs_epi = [p for p in pairs if p.get('epitope', '') in val_epitopes]
    
    print(f"Scenario 2 (Unseen Epitopes):")
    print(f"  Train epitopes: {len(train_epitopes)}")
    print(f"  Val epitopes: {len(val_epitopes)}")
    print(f"  Train pairs: {len(train_pairs_epi)}")
    print(f"  Val pairs: {len(val_pairs_epi)}")
    
    splits_epitope = {
        'train': train_pairs_epi,
        'val': val_pairs_epi,
        'train_epitopes': train_epitopes,
        'val_epitopes': val_epitopes
    }
    
    with open(splits_dir / 'splits_epitope.json', 'w') as f:
        json.dump(splits_epitope, f, indent=2)
        
    print(f"Saved to {splits_dir / 'splits_epitope.json'}")
else:
    print("Not enough epitopes for unseen split")

## 6. Test Data Loader

In [ ]:
from src.utils.data_utils import BioPhysTCRDataset, collate_biophystcr, create_data_loaders

# Test loading a single sample
dataset = BioPhysTCRDataset(
    pairs_file=splits_dir / 'splits_random.json',
    features_dir=processed_dir
)

# Need to adjust - the dataset expects pairs directly, not a dict
# Let's create a simple test
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Save train/val pairs as separate files for the dataset
with open(splits_dir / 'train.json', 'w') as f:
    json.dump(train_pairs, f)

with open(splits_dir / 'val.json', 'w') as f:
    json.dump(val_pairs, f)

print("Saved train.json and val.json")

In [ ]:
# Now test the dataset
dataset = BioPhysTCRDataset(
    pairs_file=splits_dir / 'train.json',
    features_dir=processed_dir
)

print(f"Train dataset size: {len(dataset)}")

# Get a sample
if len(dataset) > 0:
    sample = dataset[0]
    print("\nSample structure:")
    print(f"  TCR keys: {list(sample['tcr'].keys())}")
    print(f"  pMHC keys: {list(sample['pmhc'].keys())}")
    print(f"  Label: {sample['label']}")
    print(f"  Metadata: {sample['metadata']}")
    
    print("\nTensor shapes:")
    for key, value in sample['tcr'].items():
        if isinstance(value, torch.Tensor):
            print(f"  tcr.{key}: {value.shape}")

In [ ]:
# Test DataLoader
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_biophystcr,
    num_workers=0  # Use 0 for debugging
)

# Get a batch
batch = next(iter(train_loader))

print("Batch structure:")
print(f"  TCR sequence_emb: {batch['tcr']['sequence_emb'].shape}")
print(f"  TCR structure_x: {batch['tcr']['structure_x'].shape}")
print(f"  TCR structure_edge_index: {batch['tcr']['structure_edge_index'].shape}")
print(f"  TCR structure_batch: {batch['tcr']['structure_batch'].shape}")
print(f"  TCR physchem_features: {batch['tcr']['physchem_features'].shape}")
print(f"  Labels: {batch['label'].shape}")

## 7. Test Model Forward Pass

In [ ]:
from src.models.biophystcr import BioPhysTCR, BioPhysTCRConfig

# Create model
config = BioPhysTCRConfig()
model = BioPhysTCR(config)

print(f"Model created with config:")
print(f"  ESM2 dim: {config.esm2_dim}")
print(f"  SaProt dim: {config.saprot_dim}")
print(f"  Physchem dim: {config.physchem_dim}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Test forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Move batch to device
tcr_data = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
            for k, v in batch['tcr'].items()}
pmhc_data = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
             for k, v in batch['pmhc'].items()}

# Forward pass
with torch.no_grad():
    outputs = model(tcr_data, pmhc_data)

print("Model outputs:")
for key, value in outputs.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

## 8. Summary

If all cells ran successfully, your data is ready for training!

**Next steps:**
1. Run `02_train_model.ipynb` for Scenario 1 training
2. Run `03_evaluation.ipynb` for evaluation and analysis

In [ ]:
print("Data preparation complete!")
print("\nFiles created:")
print(f"  - {splits_dir / 'train.json'}")
print(f"  - {splits_dir / 'val.json'}")
print(f"  - {splits_dir / 'splits_random.json'}")
if (splits_dir / 'splits_epitope.json').exists():
    print(f"  - {splits_dir / 'splits_epitope.json'}")